# 07 · Construcción del dataset multimodal definitivo del recomendador

PAra este notebook se usan tres archivos principales:

- *neighborhood_real_estate_enriched.csv*: una fila por barrio con la información inmobiliaria consolidada.
- *transport_routes_by_neighborhood.csv*: una fila por combinación **barrio × destino × modo de transporte**, incluyendo los estados de ida y vuelta y las métricas diarias.
- *transport_route_requests_audit.csv*: una fila por consulta individual a OpenTripPlanner, identificada por **fecha × barrio × destino × escenario × modo**.

El objetivo de este notebook es dejar preparado un dataset limpio y consistente para usarlo después, sin eliminar las combinaciones en las que OpenTripPlanner no ha encontrado una ruta.

Se generan dos versiones:

- **Dataset completo****: todas las combinaciones de barrio, destino y modo de transporte.
- **Dataset para Pareto y KNN**: solo con los casos que tienen una ruta completa, un precio válido y un tiempo diario válido.

**ACLARACION:** El modo de transporte se mantiene como una característica más de cada alternativa y como una preferencia que podrá elegir el usuario más adelante; no se trata como un objetivo adicional en Pareto.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)


In [3]:
# Claves principales.
JOIN_KEY = "zone_key"
DESTINATION_KEY = "destination_id"
SCENARIO_KEY = "scenario_id"
MODE_KEY = "transport_mode"
SERVICE_DATE_KEY = "service_date"
PRICE_COLUMN = "price_m2_for_recommender"

ALTERNATIVE_KEY = [
    JOIN_KEY,
    DESTINATION_KEY,
    MODE_KEY,
]

AUDIT_LOGICAL_KEY = [
    SERVICE_DATE_KEY,
    JOIN_KEY,
    DESTINATION_KEY,
    SCENARIO_KEY,
    MODE_KEY,
]

MORNING_SCENARIO = "morning_arrival_0900"
EVENING_SCENARIO = "evening_departure_1800"

EXPECTED_SCENARIOS = {
    MORNING_SCENARIO,
    EVENING_SCENARIO,
}


## 1. Localización de los ficheros

Se buscan automáticamente las carpetas principales del proyecto. Las rutas de entrada reflejan los nombres generados por la versión multimodal del notebook 06.


In [4]:
def find_project_root(start: Path | None = None) -> Path:
    """Busca la carpeta principal del proyecto."""

    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "data").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "No se ha encontrado la carpeta principal del proyecto. "
        "Debe contener las carpetas data/ y notebooks/."
    )


In [5]:
PROJECT_ROOT = find_project_root()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FINAL_DIR = PROJECT_ROOT / "data" / "final"

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

REAL_ESTATE_PATH = (
    PROCESSED_DIR
    / "neighborhood_real_estate_enriched.csv"
)

TRANSPORT_PATH = (
    PROCESSED_DIR
    / "transport_routes_by_neighborhood.csv"
)

TRANSPORT_AUDIT_PATH = (
    PROCESSED_DIR
    / "transport_route_requests_audit.csv"
)

FINAL_DATASET_PATH = (
    FINAL_DIR
    / "neighborhood_recommender_dataset.csv"
)

MODEL_READY_PATH = (
    FINAL_DIR
    / "neighborhood_recommender_model_ready.csv"
)

NO_ROUTE_PATH = (
    FINAL_DIR
    / "transport_no_route_requests.csv"
)

MISSING_REPORT_PATH = (
    FINAL_DIR
    / "missing_values_report.csv"
)


In [6]:
required_paths = [
    REAL_ESTATE_PATH,
    TRANSPORT_PATH,
    TRANSPORT_AUDIT_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    formatted_paths = "\n".join(
        f"- {path}"
        for path in missing_paths
    )

    raise FileNotFoundError(
        "Faltan los siguientes archivos:\n"
        f"{formatted_paths}\n\n"
        "Comprueba que el notebook 06 multimodal "
        "se ha ejecutado y guardado correctamente."
    )


In [7]:
print("Raíz del proyecto:", PROJECT_ROOT)
print("Datos inmobiliarios:", REAL_ESTATE_PATH)
print("Resumen multimodal:", TRANSPORT_PATH)
print("Auditoría de transporte:", TRANSPORT_AUDIT_PATH)
print("Carpeta de salida:", FINAL_DIR)


Raíz del proyecto: /home/aclaver/TFG-INFO
Datos inmobiliarios: /home/aclaver/TFG-INFO/data/processed/neighborhood_real_estate_enriched.csv
Resumen multimodal: /home/aclaver/TFG-INFO/data/processed/transport_routes_by_neighborhood.csv
Auditoría de transporte: /home/aclaver/TFG-INFO/data/processed/transport_route_requests_audit.csv
Carpeta de salida: /home/aclaver/TFG-INFO/data/final


## 2. Carga de los datasets

La auditoría sirve simplemente para comprobar que lo que aparece guardado en ese resumen coincide con las consultas que realmente se hicieron en *OpenTripPlanner*.


In [8]:
real_estate = pd.read_csv(
    REAL_ESTATE_PATH,
)

transport = pd.read_csv(
    TRANSPORT_PATH,
)

transport_audit = pd.read_csv(
    TRANSPORT_AUDIT_PATH,
)


In [9]:
print(
    "Dimensiones del dataset inmobiliario:",
    real_estate.shape,
)

print(
    "Dimensiones del resumen multimodal:",
    transport.shape,
)

print(
    "Dimensiones de la auditoría:",
    transport_audit.shape,
)



Dimensiones del dataset inmobiliario: (131, 103)
Dimensiones del resumen multimodal: (1572, 49)
Dimensiones de la auditoría: (3144, 23)


In [10]:
display(real_estate.head())

,zone_key,neighborhood_name,district_name,has_real_estate_data,price_estimate_available,coverage_status,total_listings,eligible_listings,excluded_listings,eligible_percentage,excluded_percentage,market_share_percentage,num_listings,price_eur_mean,price_eur_median,price_eur_q1,price_eur_q3,price_eur_iqr,price_eur_min,price_eur_max,price_m2_mean,price_m2_median,price_m2_q1,price_m2_q3,price_m2_iqr,price_m2_std,price_m2_relative_iqr_percentage,price_m2_median_ci_low,price_m2_median_ci_high,price_m2_median_ci_width,price_m2_median_ci_relative_width_percentage,price_m2_median_bootstrap_se,bootstrap_sample_size,bootstrap_iterations,bootstrap_confidence_level,bootstrap_status,area_m2_mean,area_m2_median,area_m2_q1,area_m2_q3,area_m2_iqr,area_m2_min,area_m2_max,rooms_mean,rooms_median,rooms_known_count,rooms_unknown_count,rooms_coverage_percentage,bathrooms_mean,bathrooms_median,bathrooms_known_count,bathrooms_unknown_count,bathrooms_coverage_percentage,elevator_ratio,elevator_percentage,elevator_known_count,elevator_unknown_count,elevator_coverage_percentage,exterior_ratio,exterior_percentage,exterior_known_count,exterior_unknown_count,exterior_coverage_percentage,sample_size_score,ci_precision_score,price_homogeneity_score,price_reliability_score,price_reliability_level,price_reliability_reason,market_scope,price_type,source_snapshot_date_status,registered_price_m2_total_2025,registered_price_m2_new_2025,registered_price_m2_used_2025,registered_index_madrid_total_2025,registered_index_madrid_new_2025,registered_index_madrid_used_2025,registered_index_type_total_2025,registered_index_type_new_2025,registered_index_type_used_2025,registered_price_available,registered_new_price_available,registered_used_price_available,reference_year,price_unit,price_statistic,housing_market_scope,source_publisher,source_origin,source_series_code,minimum_cases_to_publish_neighborhood,source_coverage_2025_percentage,kaggle_asking_price_available,kaggle_features_available,kaggle_sample_reliability_score,kaggle_sample_reliability_level,price_m2_for_recommender,price_m2_for_recommender_source,housing_data_availability_status,registered_minus_kaggle_eur_m2,registered_vs_kaggle_gap_percentage,registered_to_kaggle_price_ratio
0,MAD-022,Acacias,Arganzuela,True,True,available,67,66,1,98.51,1.49,0.59,66,400937.86,399500.0,213500.00,490000.0,276500.00,108000.0,1000000.0,6121.14,6020.38,5282.58,7200.00,1917.42,1362.98,31.85,5357.14,6282.05,924.91,15.36,281.15,66,2000,0.95,calculated,71.56,63.0,35.00,86.25,51.25,12.0,280.0,2.25,2.0,52,14,78.79,1.75,2.0,12,54,18.18,0.7344,73.44,64,2,96.97,0.6562,65.62,64,2,96.97,50.79,45.24,47.22,47.68,low,La estimación debe interpretarse con precaució...,residential_sale,asking_price,unknown,5736.45,NaN,5740.58,108.53,NaN,107.63,100,NaN,100.07,True,False,True,2025,EUR_per_m2,mean_declared_transaction_price,residential_sales,Ayuntamiento de Madrid,Colegio de Registradores de España,504020100060,15,97.7,True,True,47.68,low,5736.45,registered_transactions_2025,official_price_and_kaggle_features,-283.93,-4.716147,0.952839
1,MAD-027,Atocha,Arganzuela,False,False,no_listings,0,0,0,NaN,NaN,0.00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,2000,0.95,no_data,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,no_data,No hay anuncios elegibles para calcular indica...,residential_sale,asking_price,unknown,6810.36,6828.78,NaN,128.84,135.45,NaN,100,2.56,NaN,True,True,False,2025,EUR_per_m2,mean_declared_transaction_price,residential_sales,Ayuntamiento de Madrid,Colegio de Registradores de España,504020100060,15,97.7,False,False,NaN,no_data,6810.36,registered_transactions_2025,official_price_only,NaN,NaN,NaN
2,MAD-023,Chopera,Arganzuela,True,True,available,35,34,1,97.14,2.86,0.31,34,420111.47,399950.0,332500.00,509000.0,176500.00,180000.0,749900.0,5168.28,5165.79,4795.05,5528.20,733.15,652.15,14.19,4865.02,5415.78,550.76,10.66,141.68,34,2000,0.95,calculated

In [11]:
display(transport.head())

,zone_key,neighborhood_name,district_name,destination_id,destination_name,transport_mode,transport_mode_label,service_date,morning_requested_datetime,morning_route_status,morning_routing_errors_json,morning_request_error,evening_requested_datetime,evening_route_status,evening_routing_errors_json,evening_request_error,morning_start_datetime,morning_end_datetime,morning_all_modes,morning_transit_modes,morning_routes_used,morning_agencies_used,morning_duration_minutes,morning_distance_meters,morning_walk_time_minutes,morning_waiting_time_minutes,morning_walk_distance_meters,morning_number_of_transfers,evening_start_datetime,evening_end_datetime,evening_all_modes,evening_transit_modes,evening_routes_used,evening_agencies_used,evening_duration_minutes,evening_distance_meters,evening_walk_time_minutes,evening_waiting_time_minutes,evening_walk_distance_meters,evening_number_of_transfers,route_status,route_available,no_route_scenarios,commute_daily_minutes,commute_daily_distance_meters,commute_daily_walk_minutes,commute_daily_waiting_minutes,commute_daily_walk_distance_meters,commute_daily_transfers
0,MAD-011,Palacio,Centro,DEST_001,Puerta del Sol,BICYCLE,Bicicleta,2026-07-29,2026-07-29T09:00:00+02:00,success,[],NaN,2026-07-29T18:00:00+02:00,success,[],NaN,2026-07-29T08:51:56+02:00,2026-07-29T09:00:00+02:00,BICYCLE,NaN,NaN,NaN,8.066667,1216.02,0.000000,0.0,0.00,0.0,2026-07-29T18:00:00+02:00,2026-07-29T18:07:40+02:00,BICYCLE,NaN,NaN,NaN,7.666667,1203.25,0.0,0.0,0.00,0.0,complete_route,True,0,15.733333,2419.27,0.000000,0.0,0.0,0.0
1,MAD-011,Palacio,Centro,DEST_001,Puerta del Sol,CAR,Coche,2026-07-29,2026-07-29T09:00:00+02:00,success,[],NaN,2026-07-29T18:00:00+02:00,success,[],NaN,2026-07-29T08:51:46+02:00,2026-07-29T09:00:00+02:00,CAR,NaN,NaN,NaN,8.233333,3518.59,0.000000,0.0,0.00,0.0,2026-07-29T18:00:00+02:00,2026-07-29T18:07:59+02:00,CAR,NaN,NaN,NaN,7.983333,2981.15,0.0,0.0,0.00,0.0,complete_route,True,0,16.216667,6499.74,0.000000,0.0,0.0,0.0
2,MAD-011,Palacio,Centro,DEST_001,Puerta del Sol,TRANSIT,Transporte público,2026-07-29,2026-07-29T09:00:00+02:00,no_route,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri...",NaN,2026-07-29T18:00:00+02:00,no_route,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_route,False,2,NaN,NaN,NaN,NaN,NaN,NaN
3,MAD-011,Palacio,Centro,DEST_001,Puerta del Sol,WALK,A pie,2026-07-29,2026-07-29T09:00:00+02:00,success,[],NaN,2026-07-29T18:00:00+02:00,success,[],NaN,2026-07-29T08:42:29+02:00,2026-07-29T09:00:00+02:00,WALK,NaN,NaN,NaN,17.516667,1203.25,17.516667,0.0,1203.25,0.0,2026-07-29T18:00:00+02:00,2026-07-29T18:17:24+02:00,WALK,NaN,NaN,NaN,17.400000,1203.25,17.4,0.0,1203.25,0.0,complete_route,True,0,34.916667,2406.50,34.916667,0.0,2406.5,0.0
4,MAD-012,Embajadores,Centro,DEST_001,Puerta del Sol,BICYCLE,Bicicleta,2026-07-29,2026-07-29T09:00:00+02:00,success,[],NaN,2026-07-29T18:00:00+02:00,success,[],NaN,2026-07-29T08:46:47+02:00,2026-07-29T09:00:00+02:00,BICYCLE,NaN,NaN,NaN,13.216667,1709.43,0.000000,0.0,0.00,0.0,2026-07-29T18:00:00+02:00,2026-07-29T18:09:58+02:00,BICYCLE,NaN,NaN,NaN,9.966667,1618.61,0.0,0.0,0.00,0.0,complete_route,True,0,23.183333,3328.04,0.000000,0.0,0.0,0.0


In [12]:
display(transport_audit.head())

,request_key,zone_key,neighborhood_name,district_name,origin_latitude,origin_longitude,destination_id,destination_name,target_latitude,target_longitude,scenario_id,scenario_name,direction,transport_mode,transport_mode_label,service_date,requested_datetime,calculation_timestamp,request_status,itineraries_found,routing_errors_json,request_error,elapsed_seconds
0,2026-07-29|MAD-011|DEST_001|morning_arrival_09...,MAD-011,Palacio,Centro,40.414971,-3.715596,DEST_001,Puerta del Sol,40.416775,-3.703790,morning_arrival_0900,Llegada al destino a las 09:00,to_destination,TRANSIT,Transporte público,2026-07-29,2026-07-29T09:00:00+02:00,2026-08-12T10:35:30.390776+02:00,no_route,0,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri...",NaN,1.026643
1,2026-07-29|MAD-011|DEST_001|morning_arrival_09...,MAD-011,Palacio,Centro,40.414971,-3.715596,DEST_001,Puerta del Sol,40.416775,-3.703790,morning_arrival_0900,Llegada al destino a las 09:00,to_destination,WALK,A pie,2026-07-29,2026-07-29T09:00:00+02:00,2026-08-12T10:35:31.417659+02:00,success,1,[],NaN,0.019682
2,2026-07-29|MAD-011|DEST_001|morning_arrival_09...,MAD-011,Palacio,Centro,40.414971,-3.715596,DEST_001,Puerta del Sol,40.416775,-3.703790,morning_arrival_0900,Llegada al destino a las 09:00,to_destination,BICYCLE,Bicicleta,2026-07-29,2026-07-29T09:00:00+02:00,2026-08-12T10:35:31.437577+02:00,success,1,[],NaN,0.020907
3,2026-07-29|MAD-011|DEST_001|morning_arrival_09...,MAD-011,Palacio,Centro,40.414971,-3.715596,DEST_001,Puerta del Sol,40.416775,-3.703790,morning_arrival_0900,Llegada al destino a las 09:00,to_destination,CAR,Coche,2026-07-29,2026-07-29T09:00:00+02:00,2026-08-12T10:35:31.459217+02:00,success,1,[],NaN,0.024474
4,2026-07-29|MAD-011|DEST_001|evening_departure_...,MAD-011,Palacio,Centro,40.416775,-3.703790,DEST_001,Puerta del Sol,40.414971,-3.715596,evening_departure_1800,Salida del destino a las 18:00,from_destination,TRANSIT,Transporte público,2026-07-29,2026-07-29T18:00:00+02:00,2026-08-12T10:35:31.483896+02:00,no_route,0,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri...",NaN,2.319081


### Estructura esperada

Los tres archivos tienen granularidades distintas:

- **Inmobiliario:** una fila por barrio.
- **Resumen multimodal:** una fila por barrio–destino–modo.
- **Auditoría:** dos consultas por barrio–destino–modo, una para la ida de mañana y otra para la vuelta de tarde.

El resumen del notebook 06 ya conserva también las combinaciones *no_route*. Por tanto, en este notebook no es necesario reconstruir artificialmente las filas ausentes: únicamente se comprueba que *resumen y auditoría* sean coherentes.


## 3. Validaciones previas de estructura


In [13]:
def require_columns(
    df: pd.DataFrame,
    columns: list[str],
    df_name: str,
) -> None:
    """Comprueba que un dataset contiene las columnas necesarias."""

    missing_columns = [
        column
        for column in columns
        if column not in df.columns
    ]

    if missing_columns:
        raise KeyError(
            f"Faltan columnas en '{df_name}': "
            f"{missing_columns}"
        )


def validate_key_values(
    df: pd.DataFrame,
    columns: list[str],
    df_name: str,
) -> None:
    """Comprueba que las columnas de clave no tienen valores nulos o vacíos."""

    for column in columns:
        if df[column].isna().any():
            raise ValueError(
                f"Hay valores ausentes en '{column}' "
                f"dentro de '{df_name}'."
            )

        if (
            df[column]
            .astype("string")
            .str.strip()
            .eq("")
            .any()
        ):
            raise ValueError(
                f"Hay valores vacíos en '{column}' "
                f"dentro de '{df_name}'."
            )


In [14]:
REQUIRED_TRANSPORT_COLUMNS = [
    *ALTERNATIVE_KEY,
    "transport_mode_label",
    SERVICE_DATE_KEY,
    "morning_route_status",
    "evening_route_status",
    "route_status",
    "route_available",
    "no_route_scenarios",
    "morning_duration_minutes",
    "evening_duration_minutes",
    "commute_daily_minutes",
]

REQUIRED_AUDIT_COLUMNS = [
    "request_key",
    *AUDIT_LOGICAL_KEY,
    "transport_mode_label",
    "request_status",
    "routing_errors_json",
    "request_error",
]

require_columns(
    real_estate,
    [
        JOIN_KEY,
        PRICE_COLUMN,
    ],
    "datos inmobiliarios",
)

require_columns(
    transport,
    REQUIRED_TRANSPORT_COLUMNS,
    "resumen multimodal",
)

require_columns(
    transport_audit,
    REQUIRED_AUDIT_COLUMNS,
    "auditoría de transporte",
)


In [15]:
# Normalización de las claves textuales.
real_estate[JOIN_KEY] = (
    real_estate[JOIN_KEY]
    .astype("string")
    .str.strip()
)

for column in [
    *ALTERNATIVE_KEY,
    "transport_mode_label",
    SERVICE_DATE_KEY,
    "morning_route_status",
    "evening_route_status",
    "route_status",
]:
    transport[column] = (
        transport[column]
        .astype("string")
        .str.strip()
    )

for column in [
    "request_key",
    *AUDIT_LOGICAL_KEY,
    "request_status",
]:
    transport_audit[column] = (
        transport_audit[column]
        .astype("string")
        .str.strip()
    )

transport["morning_route_status"] = (
    transport["morning_route_status"]
    .str.lower()
)

transport["evening_route_status"] = (
    transport["evening_route_status"]
    .str.lower()
)

transport["route_status"] = (
    transport["route_status"]
    .str.lower()
)

transport_audit["request_status"] = (
    transport_audit["request_status"]
    .str.lower()
)


In [16]:
validate_key_values(
    real_estate,
    [JOIN_KEY],
    "datos inmobiliarios",
)

validate_key_values(
    transport,
    [
        *ALTERNATIVE_KEY,
        "transport_mode_label",
        SERVICE_DATE_KEY,
    ],
    "resumen multimodal",
)

validate_key_values(
    transport_audit,
    [
        "request_key",
        *AUDIT_LOGICAL_KEY,
        "request_status",
    ],
    "auditoría de transporte",
)


In [17]:
# El dataset inmobiliario debe contener una única fila por barrio.
if real_estate[JOIN_KEY].duplicated().any():
    duplicated = real_estate.loc[
        real_estate[JOIN_KEY].duplicated(
            keep=False
        )
    ].sort_values(JOIN_KEY)

    display(duplicated)

    raise ValueError(
        "Hay barrios repetidos en el dataset inmobiliario."
    )


# El resumen multimodal debe contener una única fila por barrio, destino y modo.
if transport.duplicated(
    subset=ALTERNATIVE_KEY
).any():
    duplicated = transport.loc[
        transport.duplicated(
            subset=ALTERNATIVE_KEY,
            keep=False,
        )
    ].sort_values(ALTERNATIVE_KEY)

    display(duplicated)

    raise ValueError(
        "Hay combinaciones barrio-destino-modo "
        "repetidas en el resumen multimodal."
    )


## 4. Selección y validación del experimento de transporte

La auditoría puede conservar ejecuciones de otras fechas gracias al mecanismo *RESUME* del notebook 06. Para evitar mezclar experimentos, se toma como referencia la fecha de servicio almacenada en *transport_routes_by_neighborhood.csv* y se restringe la auditoría a esa misma ejecución.


In [18]:
service_dates = sorted(
    transport[SERVICE_DATE_KEY]
    .dropna()
    .astype(str)
    .unique()
)

if len(service_dates) != 1:
    raise ValueError(
        "El resumen multimodal debe corresponder "
        "a una única fecha de servicio. "
        f"Fechas encontradas: {service_dates}"
    )

SERVICE_DATE = service_dates[0]

observed_modes = sorted(
    transport[MODE_KEY]
    .dropna()
    .astype(str)
    .unique()
)

observed_destinations = sorted(
    transport[DESTINATION_KEY]
    .dropna()
    .astype(str)
    .unique()
)

print("Fecha de servicio:", SERVICE_DATE)
print("Modos presentes:", observed_modes)
print("Destinos presentes:", observed_destinations)


Fecha de servicio: 2026-07-29
Modos presentes: ['BICYCLE', 'CAR', 'TRANSIT', 'WALK']
Destinos presentes: ['DEST_001', 'DEST_002', 'DEST_003']


In [19]:
# Se restringe la auditoría a la misma fecha, barrios, destinos y modos que aparecen en el resumen multimodal.
current_zone_keys = set(
    transport[JOIN_KEY]
)

current_destination_ids = set(
    transport[DESTINATION_KEY]
)

current_modes = set(
    transport[MODE_KEY]
)

audit_current = transport_audit.loc[
    transport_audit[SERVICE_DATE_KEY].eq(
        SERVICE_DATE
    )
    & transport_audit[JOIN_KEY].isin(
        current_zone_keys
    )
    & transport_audit[DESTINATION_KEY].isin(
        current_destination_ids
    )
    & transport_audit[MODE_KEY].isin(
        current_modes
    )
].copy()

print(
    "Consultas de auditoría del experimento actual:",
    len(audit_current),
)


Consultas de auditoría del experimento actual: 3144


In [20]:
# Cada request_key y cada clave lógica deben ser únicas.
if audit_current["request_key"].duplicated().any():
    duplicated = audit_current.loc[
        audit_current["request_key"].duplicated(
            keep=False
        )
    ].sort_values("request_key")

    display(duplicated)

    raise ValueError(
        "Hay request_key duplicadas en la auditoría actual."
    )

if audit_current.duplicated(
    subset=AUDIT_LOGICAL_KEY
).any():
    duplicated = audit_current.loc[
        audit_current.duplicated(
            subset=AUDIT_LOGICAL_KEY,
            keep=False,
        )
    ].sort_values(AUDIT_LOGICAL_KEY)

    display(duplicated)

    raise ValueError(
        "Hay consultas lógicas duplicadas "
        "en la auditoría actual."
    )


In [21]:
observed_scenarios = set(
    audit_current[
        SCENARIO_KEY
    ].dropna()
)

if observed_scenarios != EXPECTED_SCENARIOS:
    raise ValueError(
        "Los escenarios encontrados no coinciden "
        "con los esperados. "
        f"Esperados: {EXPECTED_SCENARIOS}. "
        f"Encontrados: {observed_scenarios}."
    )


scenario_counts = (
    audit_current
    .groupby(ALTERNATIVE_KEY)[
        SCENARIO_KEY
    ]
    .nunique()
)

incomplete_alternatives = (
    scenario_counts.loc[
        scenario_counts.ne(
            len(EXPECTED_SCENARIOS)
        )
    ]
)

if not incomplete_alternatives.empty:
    display(
        incomplete_alternatives
        .rename("numero_escenarios")
        .reset_index()
    )

    raise ValueError(
        "Hay alternativas barrio-destino-modo "
        "que no contienen los dos escenarios."
    )


In [22]:
# La auditoría debe contener exactamente dos consultas por cada fila del resumen multimodal.
expected_audit_keys = {
    (
        SERVICE_DATE,
        row[JOIN_KEY],
        row[DESTINATION_KEY],
        scenario,
        row[MODE_KEY],
    )
    for _, row in transport[
        ALTERNATIVE_KEY
    ].iterrows()
    for scenario in EXPECTED_SCENARIOS
}

observed_audit_keys = {
    tuple(values)
    for values in audit_current[
        AUDIT_LOGICAL_KEY
    ].itertuples(
        index=False,
        name=None,
    )
}

missing_audit_keys = sorted(
    expected_audit_keys
    - observed_audit_keys
)

unexpected_audit_keys = sorted(
    observed_audit_keys
    - expected_audit_keys
)

if missing_audit_keys:
    print(
        "Primeras consultas ausentes:",
        missing_audit_keys[:10],
    )
    raise ValueError(
        "Faltan consultas esperadas en la auditoría."
    )

if unexpected_audit_keys:
    print(
        "Primeras consultas inesperadas:",
        unexpected_audit_keys[:10],
    )
    raise ValueError(
        "La auditoría contiene consultas "
        "ajenas al resumen multimodal."
    )

print(
    "Consultas esperadas:",
    len(expected_audit_keys),
)

print(
    "Consultas auditadas:",
    len(observed_audit_keys),
)


Consultas esperadas: 3144
Consultas auditadas: 3144


In [23]:
allowed_request_statuses = {
    "success",
    "no_route",
    "error",
}

unexpected_statuses = sorted(
    set(
        audit_current[
            "request_status"
        ].dropna()
    )
    - allowed_request_statuses
)

if unexpected_statuses:
    raise ValueError(
        "Se han encontrado estados de consulta "
        f"no reconocidos: {unexpected_statuses}"
    )

technical_errors = audit_current.loc[
    audit_current[
        "request_status"
    ].eq("error")
].copy()

if not technical_errors.empty:
    display(
        technical_errors[
            [
                *AUDIT_LOGICAL_KEY,
                "request_error",
            ]
        ].head(20)
    )

    raise ValueError(
        "La ejecución contiene errores técnicos. "
        "Deben resolverse en el notebook 06 antes "
        "de construir el dataset del recomendador."
    )


In [24]:
status_by_mode = (
    audit_current
    .groupby(
        [
            MODE_KEY,
            "request_status",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "consultas"
        }
    )
)

display(status_by_mode)


,transport_mode,request_status,consultas
0,BICYCLE,no_route,6
1,BICYCLE,success,780
2,CAR,no_route,54
3,CAR,success,732
4,TRANSIT,no_route,37
5,TRANSIT,success,749
6,WALK,no_route,114
7,WALK,success,672


## 5. Consistencia entre el resumen multimodal y la auditoría

El notebook 06 ya construye *route_status*, pero aquí se vuelve a derivar a partir de la auditoría para comprobar de forma independiente que la integración no acarrea inconsistencias.

Una alternativa se considera:

- *complete_route*: ida y vuelta correctas.
- *partial_route*: solo uno de los dos trayectos está disponible.
- *no_route*: no existe ninguno de los dos trayectos.
- *technical_error*: existe al menos un error técnico; este caso se bloquea antes de continuar.


In [25]:
audit_status_wide = (
    audit_current
    .pivot(
        index=ALTERNATIVE_KEY,
        columns=SCENARIO_KEY,
        values="request_status",
    )
    .reset_index()
)

audit_status_wide.columns.name = None

audit_status_wide = (
    audit_status_wide.rename(
        columns={
            MORNING_SCENARIO:
                "audit_morning_route_status",
            EVENING_SCENARIO:
                "audit_evening_route_status",
        }
    )
)

require_columns(
    audit_status_wide,
    [
        *ALTERNATIVE_KEY,
        "audit_morning_route_status",
        "audit_evening_route_status",
    ],
    "estados pivotados de auditoría",
)


In [26]:
transport_check = (
    transport
    .merge(
        audit_status_wide,
        on=ALTERNATIVE_KEY,
        how="left",
        validate="1:1",
    )
    .copy()
)

morning_success = (
    transport_check[
        "audit_morning_route_status"
    ].eq("success")
)

evening_success = (
    transport_check[
        "audit_evening_route_status"
    ].eq("success")
)

morning_no_route = (
    transport_check[
        "audit_morning_route_status"
    ].eq("no_route")
)

evening_no_route = (
    transport_check[
        "audit_evening_route_status"
    ].eq("no_route")
)

transport_check[
    "route_status_expected"
] = np.select(
    [
        morning_success
        & evening_success,

        morning_no_route
        & evening_no_route,

        morning_success
        ^ evening_success,
    ],
    [
        "complete_route",
        "no_route",
        "partial_route",
    ],
    default="technical_error",
)

transport_check[
    "no_route_scenarios_expected"
] = (
    transport_check[
        [
            "audit_morning_route_status",
            "audit_evening_route_status",
        ]
    ]
    .eq("no_route")
    .sum(axis=1)
)


In [27]:
# Los estados de mañana y tarde almacenados en el resumen deben coincidir con la auditoría.
morning_mismatch = (
    transport_check[
        "morning_route_status"
    ].ne(
        transport_check[
            "audit_morning_route_status"
        ]
    )
)

evening_mismatch = (
    transport_check[
        "evening_route_status"
    ].ne(
        transport_check[
            "audit_evening_route_status"
        ]
    )
)

route_status_mismatch = (
    transport_check[
        "route_status"
    ].ne(
        transport_check[
            "route_status_expected"
        ]
    )
)

no_route_count_mismatch = (
    pd.to_numeric(
        transport_check[
            "no_route_scenarios"
        ],
        errors="coerce",
    )
    .ne(
        transport_check[
            "no_route_scenarios_expected"
        ]
    )
)

if (
    morning_mismatch.any()
    or evening_mismatch.any()
    or route_status_mismatch.any()
    or no_route_count_mismatch.any()
):
    inconsistent = transport_check.loc[
        morning_mismatch
        | evening_mismatch
        | route_status_mismatch
        | no_route_count_mismatch,
        [
            *ALTERNATIVE_KEY,
            "morning_route_status",
            "audit_morning_route_status",
            "evening_route_status",
            "audit_evening_route_status",
            "route_status",
            "route_status_expected",
            "no_route_scenarios",
            "no_route_scenarios_expected",
        ],
    ]

    display(inconsistent.head(30))

    raise ValueError(
        "El resumen multimodal y la auditoría "
        "no contienen los mismos estados."
    )

print(
    "Los estados del resumen multimodal "
    "coinciden con la auditoría."
)


Los estados del resumen multimodal coinciden con la auditoría.


In [28]:
# Validación explícita de route_available.
route_available_observed = (
    transport_check[
        "route_available"
    ]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(
        {
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        }
    )
)

if route_available_observed.isna().any():
    invalid_values = (
        transport_check.loc[
            route_available_observed.isna(),
            "route_available",
        ]
        .drop_duplicates()
        .tolist()
    )

    raise ValueError(
        "Hay valores no reconocidos en route_available: "
        f"{invalid_values}"
    )

route_available_expected = (
    transport_check[
        "route_status"
    ].eq("complete_route")
)

if not route_available_observed.eq(
    route_available_expected
).all():
    raise ValueError(
        "route_available no coincide con route_status."
    )


In [29]:
no_route_requests = (
    audit_current.loc[
        audit_current[
            "request_status"
        ].eq("no_route")
    ]
    .copy()
    .sort_values(AUDIT_LOGICAL_KEY)
    .reset_index(drop=True)
)

print(
    "Consultas sin itinerario:",
    len(no_route_requests),
)

if not no_route_requests.empty:
    display(
        no_route_requests[
            [
                *AUDIT_LOGICAL_KEY,
                "transport_mode_label",
                "routing_errors_json",
            ]
        ].head(20)
    )


Consultas sin itinerario: 211


,service_date,zone_key,destination_id,scenario_id,transport_mode,transport_mode_label,routing_errors_json
0,2026-07-29,MAD-011,DEST_001,evening_departure_1800,TRANSIT,Transporte público,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri..."
1,2026-07-29,MAD-011,DEST_001,morning_arrival_0900,TRANSIT,Transporte público,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri..."
2,2026-07-29,MAD-012,DEST_001,evening_departure_1800,TRANSIT,Transporte público,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri..."
3,2026-07-29,MAD-012,DEST_001,morning_arrival_0900,TRANSIT,Transporte público,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri..."
4,2026-07-29,MAD-013,DEST_001,evening_departure_1800,TRANSIT,Transporte público,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri..."
5,2026-07-29,MAD-013,DEST_001,morning_arrival_0900,TRANSIT,Transporte público,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri..."
6,2026-07-29,MAD-014,DEST_001,evening_departure_1800,TRANSIT,Transporte público,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri..."
7,2026-07-29,MAD-014,DEST_001,morning_arrival_0900,TRANSIT,Transporte público,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri..."
8,2026-07-29,MAD-015,DEST_001,evening_departure_1800,TRANSIT,Transporte público,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri..."
9,2026-07-29,MAD-015,DEST_001,morning_arrival_0900,TRANSIT,Transporte público,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri..."


## 6. Comprobación de los tiempos de desplazamiento

Los valores ausentes de transporte deben responder al estado de la ruta y no a una imputación arbitraria. Para una consulta *no_route*, las métricas del escenario correspondiente deben permanecer vacías. 

**NOTA:** *commute_daily_minutes* solo puede considerarse válido cuando existen tanto la ida como la vuelta.


In [30]:
MORNING_ROUTE_COLUMNS = [
    column
    for column in [
        "morning_duration_minutes",
        "morning_distance_meters",
        "morning_walk_time_minutes",
        "morning_waiting_time_minutes",
        "morning_walk_distance_meters",
        "morning_number_of_transfers",
    ]
    if column in transport_check.columns
]

EVENING_ROUTE_COLUMNS = [
    column
    for column in [
        "evening_duration_minutes",
        "evening_distance_meters",
        "evening_walk_time_minutes",
        "evening_waiting_time_minutes",
        "evening_walk_distance_meters",
        "evening_number_of_transfers",
    ]
    if column in transport_check.columns
]

DAILY_ROUTE_COLUMNS = [
    column
    for column in [
        "commute_daily_minutes",
        "commute_daily_distance_meters",
        "commute_daily_walk_minutes",
        "commute_daily_waiting_minutes",
        "commute_daily_walk_distance_meters",
        "commute_daily_transfers",
    ]
    if column in transport_check.columns
]

if "morning_duration_minutes" not in MORNING_ROUTE_COLUMNS:
    raise KeyError(
        "Falta morning_duration_minutes."
    )

if "evening_duration_minutes" not in EVENING_ROUTE_COLUMNS:
    raise KeyError(
        "Falta evening_duration_minutes."
    )

if "commute_daily_minutes" not in DAILY_ROUTE_COLUMNS:
    raise KeyError(
        "Falta commute_daily_minutes."
    )


In [31]:
morning_no_route_mask = (
    transport_check[
        "morning_route_status"
    ].eq("no_route")
)

evening_no_route_mask = (
    transport_check[
        "evening_route_status"
    ].eq("no_route")
)

invalid_morning_values = (
    transport_check.loc[
        morning_no_route_mask,
        MORNING_ROUTE_COLUMNS,
    ]
    .notna()
    .any(axis=1)
)

invalid_evening_values = (
    transport_check.loc[
        evening_no_route_mask,
        EVENING_ROUTE_COLUMNS,
    ]
    .notna()
    .any(axis=1)
)

if invalid_morning_values.any():
    display(
        transport_check.loc[
            morning_no_route_mask
            & invalid_morning_values,
            ALTERNATIVE_KEY
            + ["morning_route_status"]
            + MORNING_ROUTE_COLUMNS,
        ]
    )

    raise ValueError(
        "Hay rutas de mañana no encontradas "
        "que contienen métricas informadas."
    )

if invalid_evening_values.any():
    display(
        transport_check.loc[
            evening_no_route_mask
            & invalid_evening_values,
            ALTERNATIVE_KEY
            + ["evening_route_status"]
            + EVENING_ROUTE_COLUMNS,
        ]
    )

    raise ValueError(
        "Hay rutas de tarde no encontradas "
        "que contienen métricas informadas."
    )

print(
    "Las consultas no_route mantienen "
    "sus métricas de escenario como valores ausentes."
)


Las consultas no_route mantienen sus métricas de escenario como valores ausentes.


In [32]:
# Las rutas completas deben tener tiempos positivos.
complete_route_mask = (
    transport_check[
        "route_status"
    ].eq("complete_route")
)

for column in [
    "morning_duration_minutes",
    "evening_duration_minutes",
    "commute_daily_minutes",
]:
    values = pd.to_numeric(
        transport_check.loc[
            complete_route_mask,
            column,
        ],
        errors="coerce",
    )

    if values.isna().any():
        raise ValueError(
            f"Hay rutas completas sin valor válido en {column}."
        )

    if (values <= 0).any():
        raise ValueError(
            f"Hay rutas completas con valores "
            f"no positivos en {column}."
        )




In [33]:
# Las rutas no completas no deben tener un tiempo diario total.
invalid_daily_time = (
    transport_check.loc[
        ~complete_route_mask,
        "commute_daily_minutes",
    ]
    .notna()
)

if invalid_daily_time.any():
    display(
        transport_check.loc[
            ~complete_route_mask
            & transport_check[
                "commute_daily_minutes"
            ].notna(),
            ALTERNATIVE_KEY
            + [
                "route_status",
                "commute_daily_minutes",
            ],
        ]
    )

    raise ValueError(
        "Hay rutas no completas con un "
        "commute_daily_minutes informado."
    )

print(
    "La disponibilidad de los tiempos diarios "
    "es coherente con el estado de las rutas."
)


La disponibilidad de los tiempos diarios es coherente con el estado de las rutas.


## 7. Cobertura de la matriz multimodal

Antes de unir los datos inmobiliarios se comprubea que el resumen de transporte conserva la matriz completa de alternativas.

La unidad esperada es: **barrio × destino × modo de transporte**

Incluso si no existe una ruta, la combinación debe seguir apareciendo en *transport_routes_by_neighborhood.csv* con su estado correspondiente.


In [34]:
real_estate_zones = set(
    real_estate[JOIN_KEY]
)

transport_zones = set(
    transport_check[JOIN_KEY]
)

missing_transport_zones = sorted(
    real_estate_zones
    - transport_zones
)

unexpected_transport_zones = sorted(
    transport_zones
    - real_estate_zones
)

if missing_transport_zones:
    print(
        "Primeros barrios sin transporte:",
        missing_transport_zones[:20],
    )
    raise ValueError(
        "El resumen multimodal no contiene "
        "todos los barrios inmobiliarios. "
        "Comprueba que el notebook 06 se ejecutó "
        "con TEST_MODE = False."
    )

if unexpected_transport_zones:
    print(
        "Primeros barrios inesperados:",
        unexpected_transport_zones[:20],
    )
    raise ValueError(
        "El resumen multimodal contiene barrios "
        "que no aparecen en el dataset inmobiliario."
    )


In [35]:
expected_alternatives = {
    (
        zone_key,
        destination_id,
        transport_mode,
    )
    for zone_key in sorted(
        real_estate_zones
    )
    for destination_id in sorted(
        current_destination_ids
    )
    for transport_mode in sorted(
        current_modes
    )
}

observed_alternatives = {
    tuple(values)
    for values in transport_check[
        ALTERNATIVE_KEY
    ].itertuples(
        index=False,
        name=None,
    )
}

missing_alternatives = sorted(
    expected_alternatives
    - observed_alternatives
)

unexpected_alternatives = sorted(
    observed_alternatives
    - expected_alternatives
)

if missing_alternatives:
    print(
        "Primeras alternativas ausentes:",
        missing_alternatives[:20],
    )
    raise ValueError(
        "Faltan combinaciones barrio-destino-modo "
        "en el resumen multimodal."
    )

if unexpected_alternatives:
    print(
        "Primeras alternativas inesperadas:",
        unexpected_alternatives[:20],
    )
    raise ValueError(
        "Existen combinaciones barrio-destino-modo "
        "no esperadas."
    )

print(
    "Barrios:",
    len(real_estate_zones),
)

print(
    "Destinos:",
    len(current_destination_ids),
)

print(
    "Modos:",
    len(current_modes),
)

print(
    "Combinaciones barrio-destino-modo esperadas:",
    len(expected_alternatives),
)

print(
    "Combinaciones presentes:",
    len(observed_alternatives),
)


Barrios: 131
Destinos: 3
Modos: 4
Combinaciones barrio-destino-modo esperadas: 1572
Combinaciones presentes: 1572


## 8. Unión de los datos inmobiliarios y multimodales

La relación entre ambos archivos es de **uno a varios**:

- Cada barrio aparece una vez en el dataset inmobiliario;
- Cada barrio aparece varias veces en transporte, una por cada destino y modo.

Por tanto, cada fila del resultado representa una alternativa concreta **barrio–destino–modo**.


In [36]:
# Comprobación de coherencia de los nombres descriptivos antes de eliminar columnas redundantes durante la unión.
for descriptive_column in [
    "neighborhood_name",
    "district_name",
]:
    if (
        descriptive_column in real_estate.columns
        and descriptive_column in transport_check.columns
    ):
        real_description = (
            real_estate[
                [
                    JOIN_KEY,
                    descriptive_column,
                ]
            ]
            .drop_duplicates()
        )

        transport_description = (
            transport_check[
                [
                    JOIN_KEY,
                    descriptive_column,
                ]
            ]
            .drop_duplicates()
        )

        description_check = (
            real_description
            .merge(
                transport_description,
                on=JOIN_KEY,
                how="inner",
                validate="1:1",
                suffixes=(
                    "_real_estate",
                    "_transport",
                ),
            )
        )

        left_values = (
            description_check[
                f"{descriptive_column}_real_estate"
            ]
            .astype("string")
            .str.strip()
        )

        right_values = (
            description_check[
                f"{descriptive_column}_transport"
            ]
            .astype("string")
            .str.strip()
        )

        mismatch = (
            left_values.notna()
            & right_values.notna()
            & left_values.ne(right_values)
        )

        if mismatch.any():
            display(
                description_check.loc[
                    mismatch
                ].head(20)
            )

            raise ValueError(
                "Hay discrepancias entre las "
                f"descripciones de {descriptive_column}."
            )


In [37]:
redundant_transport_columns = [
    column
    for column in [
        "neighborhood_name",
        "district_name",
    ]
    if (
        column in real_estate.columns
        and column in transport_check.columns
    )
]

transport_for_merge = (
    transport_check
    .drop(
        columns=[
            *redundant_transport_columns,
            "audit_morning_route_status",
            "audit_evening_route_status",
            "route_status_expected",
            "no_route_scenarios_expected",
        ],
        errors="ignore",
    )
    .copy()
)

dataset = (
    real_estate
    .copy()
    .merge(
        transport_for_merge,
        on=JOIN_KEY,
        how="left",
        validate="1:m",
        indicator=True,
        suffixes=(
            "",
            "_transport",
        ),
    )
)

print(
    "Dimensiones después de la unión:",
    dataset.shape,
)

display(
    dataset["_merge"]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "resultado_union"
    )
    .reset_index(
        name="filas"
    )
)


Dimensiones después de la unión: (1572, 150)


,resultado_union,filas
0,both,1572
1,left_only,0
2,right_only,0


In [38]:
unmatched_real_estate = (
    dataset.loc[
        dataset["_merge"].eq(
            "left_only"
        )
    ]
    .copy()
)

if not unmatched_real_estate.empty:
    display(
        unmatched_real_estate[
            [JOIN_KEY]
        ]
        .drop_duplicates()
        .sort_values(JOIN_KEY)
    )

    raise ValueError(
        "Hay barrios inmobiliarios sin "
        "información multimodal."
    )

if not dataset["_merge"].eq("both").all():
    raise ValueError(
        "La unión contiene filas que no proceden "
        "de ambos datasets."
    )

print(
    "Todas las alternativas multimodales "
    "han encontrado su barrio inmobiliario."
)


Todas las alternativas multimodales han encontrado su barrio inmobiliario.


## 9. Preparación para Pareto y KNN

Para la optimización posterior solo son imprescindibles:

- precio inmobiliario válido
- tiempo diario de desplazamiento válido
- ruta completa de ida y vuelta

Las métricas específicas de transporte (caminata, espera o transbordos) se guardan, pero no se exigen para todos los modos. De esta forma evito descartar coche o bicicleta simplemente porque una métrica propia del transporte público no sea aplicable.

La selección del modo se realizará posteriormente como una preferencia del usuario.


In [39]:
# Conversión controlada del precio a numérico.
original_price_not_null = (
    dataset[PRICE_COLUMN]
    .notna()
)

numeric_price = pd.to_numeric(
    dataset[PRICE_COLUMN],
    errors="coerce",
)

newly_invalid_price = (
    original_price_not_null
    & numeric_price.isna()
)

if newly_invalid_price.any():
    display(
        dataset.loc[
            newly_invalid_price,
            [
                JOIN_KEY,
                PRICE_COLUMN,
            ],
        ].head(20)
    )

    raise ValueError(
        "Hay precios no nulos que no pueden "
        "convertirse a formato numérico."
    )

dataset[PRICE_COLUMN] = numeric_price

dataset["commute_daily_minutes"] = (
    pd.to_numeric(
        dataset[
            "commute_daily_minutes"
        ],
        errors="coerce",
    )
)


In [40]:
# route_available se deriva de route_status para mantener una definición única y explícita dentro del dataset final.
dataset["route_available"] = (
    dataset[
        "route_status"
    ].eq("complete_route")
)

dataset["route_missing_reason"] = (
    dataset[
        "route_status"
    ]
    .map(
        {
            "complete_route":
                pd.NA,

            "partial_route":
                "falta_una_de_las_dos_rutas",

            "no_route":
                "no_hay_ruta_en_ningun_sentido",

            "technical_error":
                "error_tecnico",
        }
    )
    .astype("string")
)


In [41]:
price_valid = (
    dataset[PRICE_COLUMN]
    .notna()
    & dataset[PRICE_COLUMN].gt(0)
)

commute_valid = (
    dataset[
        "commute_daily_minutes"
    ]
    .notna()
    & dataset[
        "commute_daily_minutes"
    ].gt(0)
)

dataset[
    "usable_for_pareto_knn"
] = (
    dataset[
        "route_available"
    ]
    & price_valid
    & commute_valid
)


dataset[
    "model_readiness_status"
] = np.select(
    [
        dataset[
            "usable_for_pareto_knn"
        ],

        ~dataset[
            "route_available"
        ],

        dataset[
            PRICE_COLUMN
        ].isna(),

        dataset[
            PRICE_COLUMN
        ].le(0),

        dataset[
            "commute_daily_minutes"
        ].isna(),

        dataset[
            "commute_daily_minutes"
        ].le(0),
    ],
    [
        "usable",
        "ruta_no_completa",
        "precio_no_disponible",
        "precio_no_valido",
        "tiempo_no_disponible",
        "tiempo_no_valido",
    ],
    default="otros_datos_ausentes",
)


In [42]:
model_readiness_summary = (
    dataset[
        "model_readiness_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis("estado")
    .reset_index(
        name="combinaciones"
    )
)

display(model_readiness_summary)


,estado,combinaciones
0,usable,1457
1,ruta_no_completa,110
2,precio_no_disponible,5


### Cobertura del dataset preparado

La cobertura se analiza por destino y modo, ya que agrupar únicamente por destino mezclaría alternativas de movilidad diferentes.

**NOTA:** El porcentaje utilizable mide qué parte de las combinaciones dispone simultáneamente de ruta completa, precio válido y tiempo diario válido.


In [43]:
coverage_by_destination_mode = (
    dataset
    .groupby(
        [
            DESTINATION_KEY,
            "destination_name",
            MODE_KEY,
            "transport_mode_label",
        ],
        as_index=False,
    )
    .agg(
        total_combinations=(
            JOIN_KEY,
            "size",
        ),

        complete_routes=(
            "route_available",
            "sum",
        ),

        usable_for_recommender=(
            "usable_for_pareto_knn",
            "sum",
        ),

        missing_price=(
            PRICE_COLUMN,
            lambda series:
                int(
                    series.isna().sum()
                ),
        ),
    )
)

coverage_by_destination_mode[
    "usable_percentage"
] = (
    100
    * coverage_by_destination_mode[
        "usable_for_recommender"
    ]
    / coverage_by_destination_mode[
        "total_combinations"
    ]
).round(2)

display(
    coverage_by_destination_mode
    .sort_values(
        [
            DESTINATION_KEY,
            MODE_KEY,
        ]
    )
    .reset_index(drop=True)
)


,destination_id,destination_name,transport_mode,transport_mode_label,total_combinations,complete_routes,usable_for_recommender,missing_price,usable_percentage
0,DEST_001,Puerta del Sol,BICYCLE,Bicicleta,131,130,129,1,98.47
1,DEST_001,Puerta del Sol,CAR,Coche,131,121,121,1,92.37
2,DEST_001,Puerta del Sol,TRANSIT,Transporte público,131,120,119,1,90.84
3,DEST_001,Puerta del Sol,WALK,A pie,131,128,128,1,97.71
4,DEST_002,Nuevos Ministerios,BICYCLE,Bicicleta,131,130,129,1,98.47
5,DEST_002,Nuevos Ministerios,CAR,Coche,131,121,121,1,92.37
6,DEST_002,Nuevos Ministerios,TRANSIT,Transporte público,131,125,124,1,94.66
7,DEST_002,Nuevos Ministerios,WALK,A pie,131,129,129,1,98.47
8,DEST_003,UC3M Campus de Leganés,BICYCLE,Bicicleta,131,130,129,1,98.47
9,DEST_003,UC3M Campus de Leganés,CAR,Coche,131,121,121,1,92.37


In [44]:
coverage_by_mode = (
    dataset
    .groupby(
        [
            MODE_KEY,
            "transport_mode_label",
        ],
        as_index=False,
    )
    .agg(
        total_combinations=(
            JOIN_KEY,
            "size",
        ),

        complete_routes=(
            "route_available",
            "sum",
        ),

        usable_for_recommender=(
            "usable_for_pareto_knn",
            "sum",
        ),
    )
)

coverage_by_mode[
    "complete_route_percentage"
] = (
    100
    * coverage_by_mode[
        "complete_routes"
    ]
    / coverage_by_mode[
        "total_combinations"
    ]
).round(2)

coverage_by_mode[
    "usable_percentage"
] = (
    100
    * coverage_by_mode[
        "usable_for_recommender"
    ]
    / coverage_by_mode[
        "total_combinations"
    ]
).round(2)

display(coverage_by_mode)


,transport_mode,transport_mode_label,total_combinations,complete_routes,usable_for_recommender,complete_route_percentage,usable_percentage
0,BICYCLE,Bicicleta,393,390,387,99.24,98.47
1,CAR,Coche,393,363,363,92.37,92.37
2,TRANSIT,Transporte público,393,373,371,94.91,94.4
3,WALK,A pie,393,336,336,85.5,85.5


## 10. Informe de valores ausentes

**No todos los valores ausentes representan un problema de calidad.**

En transporte, un valor vacío puede ser la consecuencia esperada de una ruta *no_route* o de una métrica que no resulte aplicable de la misma forma a todos los modos. Los valores ausentes inmobiliarios, en cambio, proceden de la cobertura de las fuentes originales.

Por ello, el informe se interpreta junto con *route_status*, *transport_mode* y *model_readiness_status*.


In [45]:
def missing_values_report(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """Genera un resumen de valores ausentes por columna."""

    report = pd.DataFrame(
        {
            "column":
                df.columns,

            "dtype":
                df.dtypes.astype(
                    str
                ).values,

            "missing_count":
                df.isna().sum().values,

            "missing_pct":
                (
                    df.isna().mean().values
                    * 100
                ).round(2),

            "unique_non_null":
                df.nunique(
                    dropna=True
                ).values,
        }
    )

    return (
        report
        .sort_values(
            [
                "missing_count",
                "column",
            ],
            ascending=[
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )


missing_report = (
    missing_values_report(
        dataset.drop(
            columns="_merge"
        )
    )
)

display(missing_report)


,column,dtype,missing_count,missing_pct,unique_non_null
0,evening_request_error,float64,1572,100.00,0
1,morning_request_error,float64,1572,100.00,0
2,route_missing_reason,string,1462,93.00,2
3,morning_agencies_used,str,1198,76.21,4
4,morning_routes_used,str,1198,76.21,155
...,...,...,...,...,...
147,total_listings,int64,0,0.00,97
148,transport_mode,string,0,0.00,4
149,transport_mode_label,string,0,0.00,4
150,usable_for_pareto_knn,bool[pyarrow],0,0.00,2


In [46]:
missing_by_status_mode = (
    dataset
    .groupby(
        [
            MODE_KEY,
            "route_status",
        ],
        dropna=False,
    )[
        [
            PRICE_COLUMN,
            "commute_daily_minutes",
        ]
    ]
    .agg(
        lambda series:
            int(
                series.isna().sum()
            )
    )
    .reset_index()
)

display(missing_by_status_mode)


,transport_mode,route_status,price_m2_for_recommender,commute_daily_minutes
0,BICYCLE,complete_route,3,0
1,BICYCLE,no_route,0,3
2,CAR,complete_route,0,0
3,CAR,no_route,3,24
4,CAR,partial_route,0,6
5,TRANSIT,complete_route,2,0
6,TRANSIT,no_route,0,17
7,TRANSIT,partial_route,1,3
8,WALK,complete_route,0,0
9,WALK,no_route,3,57


## 11. Validaciones finales

Antes de guardar los resultados se hacen comprobaciones estrictas sobre:

- cobertura de barrios, destinos y modos
- unicidad de la alternativa barrio–destino–modo
- consistencia entre auditoría y resumen
- conservación de las consultas *no_route*
- ausencia de tiempos inventados
- validez de precio y tiempo en las filas preparadas para los modelos
- exclusión de rutas incompletas del dataset *model_ready*


In [47]:
dataset_without_merge = (
    dataset.drop(
        columns="_merge"
    )
    .copy()
)

assert len(
    dataset_without_merge
) == len(
    transport_check
), (
    "El número de filas ha cambiado "
    "durante la unión."
)

assert not dataset_without_merge.duplicated(
    subset=ALTERNATIVE_KEY
).any(), (
    "Hay alternativas barrio-destino-modo "
    "duplicadas en el dataset integrado."
)

assert (
    set(
        dataset_without_merge[
            JOIN_KEY
        ]
    )
    == set(
        real_estate[
            JOIN_KEY
        ]
    )
), (
    "Se ha perdido algún barrio "
    "durante la integración."
)


In [48]:
# El número de consultas no_route del resumen teien que coincidir exactamente con la auditoría, sin utilizar cifras fijas.
audit_no_route_count = int(
    audit_current[
        "request_status"
    ]
    .eq("no_route")
    .sum()
)

dataset_no_route_scenarios = int(
    pd.to_numeric(
        dataset_without_merge[
            "no_route_scenarios"
        ],
        errors="coerce",
    )
    .fillna(0)
    .sum()
)

assert (
    audit_no_route_count
    == dataset_no_route_scenarios
), (
    "El número de escenarios no_route "
    "no coincide entre auditoría y dataset."
)

print(
    "Consultas no_route conservadas:",
    audit_no_route_count,
)


Consultas no_route conservadas: 211


In [49]:
# Las filas utilizables tienen que cumplir las condiciones mínimas del problema precio-tiempo.
usable_mask = (
    dataset_without_merge[
        "usable_for_pareto_knn"
    ]
)

assert dataset_without_merge.loc[
    usable_mask,
    PRICE_COLUMN,
].notna().all(), (
    "Hay filas utilizables sin precio."
)

assert (
    dataset_without_merge.loc[
        usable_mask,
        PRICE_COLUMN,
    ] > 0
).all(), (
    "Hay filas utilizables con precio "
    "igual o inferior a cero."
)

assert dataset_without_merge.loc[
    usable_mask,
    "commute_daily_minutes",
].notna().all(), (
    "Hay filas utilizables sin tiempo diario."
)

assert (
    dataset_without_merge.loc[
        usable_mask,
        "commute_daily_minutes",
    ] > 0
).all(), (
    "Hay filas utilizables con tiempo "
    "igual o inferior a cero."
)

assert dataset_without_merge.loc[
    usable_mask,
    "route_status",
].eq(
    "complete_route"
).all(), (
    "Hay rutas no completas marcadas "
    "como utilizables."
)


In [50]:
# Ninguna ruta no completa puede tener un tiempo diario válido ni entrar directamente en Pareto/KNN.
incomplete_mask = (
    dataset_without_merge[
        "route_status"
    ].ne("complete_route")
)

assert not dataset_without_merge.loc[
    incomplete_mask,
    "usable_for_pareto_knn",
].any(), (
    "Alguna ruta incompleta ha sido "
    "marcada como utilizable."
)

assert dataset_without_merge.loc[
    incomplete_mask,
    "commute_daily_minutes",
].isna().all(), (
    "Alguna ruta incompleta contiene "
    "un tiempo diario total."
)


In [51]:
# La matriz completa del dataset debe seguir coincidiendo con el producto cartesiano validado anteriormente.
final_alternatives = {
    tuple(values)
    for values in dataset_without_merge[
        ALTERNATIVE_KEY
    ].itertuples(
        index=False,
        name=None,
    )
}

assert (
    final_alternatives
    == expected_alternatives
), (
    "La matriz final barrio-destino-modo "
    "no coincide con la esperada."
)

In [52]:
# La auditoría actual debe contener exactamente dos consultas por alternativa.
assert (
    len(audit_current)
    == len(dataset_without_merge)
    * len(EXPECTED_SCENARIOS)
), (
    "La auditoría no contiene exactamente "
    "dos consultas por alternativa."
)

print(
    "Todas las validaciones finales "
    "se han superado correctamente."
)


Todas las validaciones finales se han superado correctamente.


### Resultado de las validaciones

Si esta sección termina sin excepciones, el dataset conserva la matriz multimodal completa y diferencia de forma explícita las alternativas utilizables de aquellas que deben mantenerse únicamente para auditoría.

**No se fija manualmente un número de *no_route*:** la cifra se deriva de la ejecución real del notebook 06 y se contrasta entre auditoría y resumen. Esto permite que el *pipeline* siga siendo válido aunque cambie la cobertura al incorporar nuevos modos de transporte.


## 12. Preparación de los archivos finales

Se generan cuatro archivos:

1. *neighborhood_recommender_dataset.csv*: Dataset completo con una fila por **barrio × destino × modo**. Conserva también rutas parciales y alternativas sin itinerario.

2. *neighborhood_recommender_model_ready.csv*: Subconjunto preparado para Pareto y KNN. Incluye únicamente alternativas con ruta completa, precio positivo y tiempo diario positivo.

3. *transport_no_route_requests.csv*: Consultas concretas en las que OpenTripPlanner no encontró un itinerario. La cifra se obtiene dinámicamente de la auditoría.

4. *missing_values_report.csv*: Informe de valores ausentes del dataset completo.


In [53]:
dataset_complete = (
    dataset_without_merge
    .sort_values(
        ALTERNATIVE_KEY
    )
    .reset_index(drop=True)
)

dataset_model_ready = (
    dataset_complete.loc[
        dataset_complete[
            "usable_for_pareto_knn"
        ]
    ]
    .copy()
    .sort_values(
        ALTERNATIVE_KEY
    )
    .reset_index(drop=True)
)

dataset_no_route = (
    no_route_requests
    .copy()
    .sort_values(
        AUDIT_LOGICAL_KEY
    )
    .reset_index(drop=True)
)

print(
    "Dataset completo:",
    dataset_complete.shape,
)

print(
    "Dataset para Pareto y KNN:",
    dataset_model_ready.shape,
)

print(
    "Consultas no_route:",
    dataset_no_route.shape,
)


Dataset completo: (1572, 152)
Dataset para Pareto y KNN: (1457, 152)
Consultas no_route: (211, 23)


In [54]:
# Comprobaciones específicas de los archivos a guardar.
assert not dataset_complete.duplicated(
    subset=ALTERNATIVE_KEY
).any()

assert not dataset_model_ready.duplicated(
    subset=ALTERNATIVE_KEY
).any()

assert dataset_model_ready[
    "usable_for_pareto_knn"
].all()

assert dataset_model_ready[
    "route_status"
].eq(
    "complete_route"
).all()


In [55]:
dataset_complete.to_csv(
    FINAL_DATASET_PATH,
    index=False,
    encoding="utf-8-sig",
)

dataset_model_ready.to_csv(
    MODEL_READY_PATH,
    index=False,
    encoding="utf-8-sig",
)

dataset_no_route.to_csv(
    NO_ROUTE_PATH,
    index=False,
    encoding="utf-8-sig",
)

missing_report.to_csv(
    MISSING_REPORT_PATH,
    index=False,
    encoding="utf-8-sig",
)


In [56]:
print("Archivos generados:")

for path in [
    FINAL_DATASET_PATH,
    MODEL_READY_PATH,
    NO_ROUTE_PATH,
    MISSING_REPORT_PATH,
]:
    print(
        "-",
        path.relative_to(
            PROJECT_ROOT
        ),
    )


Archivos generados:
- data/final/neighborhood_recommender_dataset.csv
- data/final/neighborhood_recommender_model_ready.csv
- data/final/transport_no_route_requests.csv
- data/final/missing_values_report.csv


## 13. Resumen de los resultados

El resumen final se calcula directamente a partir de los datos generados. De esta forma no quedan cifras antiguas escritas manualmente cuando cambian la fecha, la cobertura o los modos de transporte.


In [57]:
summary = pd.Series(
    {
        "fecha_servicio":
            SERVICE_DATE,

        "barrios_inmobiliarios":
            real_estate[
                JOIN_KEY
            ].nunique(),

        "destinos_analizados":
            dataset_complete[
                DESTINATION_KEY
            ].nunique(),

        "modos_transporte":
            dataset_complete[
                MODE_KEY
            ].nunique(),

        "alternativas_barrio_destino_modo":
            len(
                dataset_complete
            ),

        "consultas_realizadas_a_otp":
            len(
                audit_current
            ),

        "consultas_correctas":
            int(
                audit_current[
                    "request_status"
                ]
                .eq("success")
                .sum()
            ),

        "consultas_no_route":
            int(
                audit_current[
                    "request_status"
                ]
                .eq("no_route")
                .sum()
            ),

        "errores_tecnicos":
            int(
                audit_current[
                    "request_status"
                ]
                .eq("error")
                .sum()
            ),

        "alternativas_con_ruta_completa":
            int(
                dataset_complete[
                    "route_status"
                ]
                .eq("complete_route")
                .sum()
            ),

        "alternativas_con_ruta_parcial":
            int(
                dataset_complete[
                    "route_status"
                ]
                .eq("partial_route")
                .sum()
            ),

        "alternativas_sin_ruta":
            int(
                dataset_complete[
                    "route_status"
                ]
                .eq("no_route")
                .sum()
            ),

        "alternativas_model_ready":
            len(
                dataset_model_ready
            ),
    }
)

display(
    summary.to_frame(
        "valor"
    )
)


,valor
fecha_servicio,2026-07-29
barrios_inmobiliarios,131
destinos_analizados,3
modos_transporte,4
alternativas_barrio_destino_modo,1572
consultas_realizadas_a_otp,3144
consultas_correctas,2933
consultas_no_route,211
errores_tecnicos,0
alternativas_con_ruta_completa,1462


In [58]:
summary_by_mode = (
    dataset_complete
    .groupby(
        [
            MODE_KEY,
            "transport_mode_label",
        ],
        as_index=False,
    )
    .agg(
        alternativas=(
            JOIN_KEY,
            "size",
        ),

        rutas_completas=(
            "route_available",
            "sum",
        ),

        alternativas_model_ready=(
            "usable_for_pareto_knn",
            "sum",
        ),

        tiempo_diario_mediano_min=(
            "commute_daily_minutes",
            "median",
        ),
    )
)

summary_by_mode[
    "cobertura_model_ready_pct"
] = (
    100
    * summary_by_mode[
        "alternativas_model_ready"
    ]
    / summary_by_mode[
        "alternativas"
    ]
).round(2)

summary_by_mode[
    "tiempo_diario_mediano_min"
] = (
    summary_by_mode[
        "tiempo_diario_mediano_min"
    ]
    .round(2)
)

display(summary_by_mode)


,transport_mode,transport_mode_label,alternativas,rutas_completas,alternativas_model_ready,tiempo_diario_mediano_min,cobertura_model_ready_pct
0,BICYCLE,Bicicleta,393,390,387,93.13,98.47
1,CAR,Coche,393,363,363,27.93,92.37
2,TRANSIT,Transporte público,393,373,371,136.77,94.4
3,WALK,A pie,393,336,336,213.48,85.5


In [59]:
assert (
    summary[
        "alternativas_barrio_destino_modo"
    ]
    == len(
        expected_alternatives
    )
)

assert (
    summary[
        "consultas_realizadas_a_otp"
    ]
    == (
        summary[
            "alternativas_barrio_destino_modo"
        ]
        * len(
            EXPECTED_SCENARIOS
        )
    )
)

assert (
    summary[
        "errores_tecnicos"
    ]
    == 0
)

print(
    "El dataset definitivo se ha construido "
    "sin eliminar las alternativas no_route "
    "ni inventar tiempos de desplazamiento."
)


El dataset definitivo se ha construido sin eliminar las alternativas no_route ni inventar tiempos de desplazamiento.


## 14. Conclusiones

A partir de aquí, cada alternativa se define como barrio–destino–modo de transporte, lo que permite comparar un mismo barrio usando distintas formas de moverse.

La auditoría del notebook 06 se usa para comprobar que las rutas y los casos *no_route* coinciden con los resultados guardados. Las combinaciones sin ruta se mantienen tal cual y no se rellenan con valores inventados.

Para **Pareto y KNN** solo se utilizan las alternativas que tienen ruta completa, precio válido y tiempo diario válido.

Así, los siguientes notebooks podrán usar el modo de transporte como una preferencia del usuario, manteniendo la comparación principal entre coste de vivienda y tiempo de desplazamiento.